# Assignment 02


# 反向传播（Backpropagation）

## 一、 符号定义与前向传播公式

假设我们有 $N$ 个样本，每个样本有 $D$ 个特征；隐藏层有 $H$ 个神经元；输出层是二分类，有 $O=1$ 个神经元。
在我们这道题中：$N = 10, D = 784, H = 16, O = 1$。

**1. 变量与权重定义：**
*   **$X$** (输入数据): `input_vector`，形状为 $(N, D)$
*   **$Y$** (真实标签): `gt_y`，形状为 $(N, 1)$
*   **$W_1$** (第一层权重): `MLP_layer_1`，形状为 $(D, H)$
*   **$W_2$** (第二层权重): `MLP_layer_2`，形状为 $(H, 1)$

**2. 激活函数（Sigmoid）：**
$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$
重要性质：Sigmoid 函数的导数：
$$ \sigma'(x) = \sigma(x) \cdot (1 - \sigma(x)) $$

**3. 前向传播（Forward Pass）数学公式：**
1.  第一层线性计算： $Z_1 = X W_1$ ，形状 $(N, H)$
2.  第一层激活： $A_1 = \sigma(Z_1)$ ，形状 $(N, H)$
3.  第二层线性计算： $Z_2 = A_1 W_2$ ，形状 $(N, 1)$
4.  第二层激活(预测值)： $\hat{Y} = A_2 = \sigma(Z_2)$ ，形状 $(N, 1)$

**4. 损失函数（二元交叉熵 Binary Cross-Entropy, BCE）：**
对所有 $N$ 个样本求和的总体 Loss 定义为 $L$（这里对应代码里的 `.sum()`）：
$$ L = -\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right] $$

---

## 二、 严谨的反向传播数学推导

我们的最终目标是求出标量损失 $L$ 对权重矩阵 $W_1$ 和 $W_2$ 的梯度，即 $\frac{\partial L}{\partial W_1}$ 和 $\frac{\partial L}{\partial W_2}$。我们利用链式法则从后往前推。

### 第一步：求损失 $L$ 对输出层未激活值 $Z_2$ 的导数 $\frac{\partial L}{\partial Z_2}$
这是整个推导中最巧妙的一步。当我们使用 Sigmoid激活函数 + 交叉熵损失函数 时，它们俩求导抵消后，最终输出层的误差梯度公式极其简单：输出层的误差梯度 = 预测值 - 真实值。下面我们针对单个样本 $i$ 来推导：

1. **求 Loss 对预测值 $\hat{y}_i$ 的偏导：**
   $$ \frac{\partial L}{\partial \hat{y}_i} = -\left( \frac{y_i}{\hat{y}_i} - \frac{1 - y_i}{1 - \hat{y}_i} \right) = \frac{\hat{y}_i - y_i}{\hat{y}_i(1 - \hat{y}_i)} $$

2. **求预测值 $\hat{y}_i$ 对未激活值 $z_{2,i}$ 的偏导（也就是Sigmoid的导数）：**
   因为 $\hat{y}_i = \sigma(z_{2,i})$，根据前面提到的性质：
   $$ \frac{\partial \hat{y}_i}{\partial z_{2,i}} = \hat{y}_i(1 - \hat{y}_i) $$

3. **使用链式法则求 $\frac{\partial L}{\partial z_{2,i}}$：**
   $$ \frac{\partial L}{\partial z_{2,i}} = \frac{\partial L}{\partial \hat{y}_i} \cdot \frac{\partial \hat{y}_i}{\partial z_{2,i}} = \left( \frac{\hat{y}_i - y_i}{\hat{y}_i(1 - \hat{y}_i)} \right) \cdot \left( \hat{y}_i(1 - \hat{y}_i) \right) $$
   分母和右边项完美约掉，剩下：
   $$ \frac{\partial L}{\partial z_{2,i}} = \hat{y}_i - y_i $$

将所有样本写成矩阵形式（用 $dZ_2$ 表示 $\frac{\partial L}{\partial Z_2}$）：
$$ dZ_2 = \hat{Y} - Y $$
*(形状：(N, 1) - (N, 1) = (N, 1))*

> **对应代码:** `dZ2 = pred_y - gt_y`

### 第二步：求损失 $L$ 对第二层权重 $W_2$ 的梯度 $\frac{\partial L}{\partial W_2}$
已知 $Z_2 = A_1 W_2$。
在线性代数矩阵求导中，如果 $Z = A W$，那么对 $W$ 的梯度为前置变量的转置乘以后置误差：$\frac{\partial L}{\partial W} = A^T \frac{\partial L}{\partial Z}$。

所以对 $W_2$ 的导数为（用 $dW_2$ 表示）：
$$ dW_2 = A_1^T dZ_2 $$
*维度检查*：$A_1^T$ 是 $(H, N)$，$dZ_2$ 是 $(N, 1)$。$(H, N) \times (N, 1) \rightarrow (H, 1)$，完美匹配 $W_2$ 的形状！

> **对应代码:** `grad_layer_2 = output_layer_1_act.T.dot(dZ2)`

### 第三步：将误差反向传播到第一层的未激活值 $Z_1$

1. **求 Loss 对第一层激活值 $A_1$ 的导数 $\frac{\partial L}{\partial A_1}$ (记为 $dA_1$)：**
   已知 $Z_2 = A_1 W_2$。对 $A_1$ 求导时，$W_2$ 是常数矩阵。
   矩阵求导法则：$\frac{\partial L}{\partial A} = \frac{\partial L}{\partial Z} W^T$。
   $$ dA_1 = dZ_2 W_2^T $$
   *维度检查*：$dZ_2$ 是 $(N, 1)$，$W_2^T$ 是 $(1, H)$。$(N, 1) \times (1, H) \rightarrow (N, H)$，匹配 $A_1$ 的形状。
   > **对应代码:** `dA1 = dZ2.dot(MLP_layer_2.T)`

2. **求 Loss 对第一层未激活值 $Z_1$ 的导数 $\frac{\partial L}{\partial Z_1}$ (记为 $dZ_1$)：**
   已知 $A_1 = \sigma(Z_1)$。
   根据链式法则，$dZ_1$ 等于 $dA_1$ 乘上激活函数的导数。这里因为是针对矩阵每个元素独立求激活函数，所以这里使用**Hadamard乘积（逐元素相乘，符号为 $\odot$）**。
   $$ dZ_1 = dA_1 \odot \sigma'(Z_1) = dA_1 \odot \left( A_1 \odot (1 - A_1) \right) $$
   *维度检查*：$(N, H) \odot (N, H) \rightarrow (N, H)$，匹配 $Z_1$ 的形状。
   > **对应代码:** `dZ1 = dA1 * output_layer_1_act * (1 - output_layer_1_act)`  （注意这里是 `*` 不是 `.dot`）

### 第四步：求损失 $L$ 对第一层权重 $W_1$ 的梯度 $\frac{\partial L}{\partial W_1}$
已知第一层的线性关系是 $Z_1 = X W_1$。
套用第二步相同的矩阵求导法则：
$$ dW_1 = X^T dZ_1 $$
*维度检查*：$X^T$ 是 $(D, N)$，$dZ_1$ 是 $(N, H)$。$(D, N) \times (N, H) \rightarrow (D, H)$，也就是 $(784, 16)$，完美匹配 $W_1$ 的形状！

> **对应代码:** `grad_layer_1 = input_vector.T.dot(dZ1)`